# 07 — Task 2.4: Instruction-based LLM Prompting
Evaluates 3 prompt strategies on a **200-sample stratified subset** of the test set using the OpenRouter API.

Set your API key:  
```bash
export OPENROUTER_API_KEY="your-key-here"
```
Or create a `.env` file at the project root with `OPENROUTER_API_KEY=...`

In [ ]:
import sys, os, time, json, re
sys.path.insert(0, '..')

import pandas as pd
from openai import OpenAI  # OpenRouter is OpenAI-compatible
from tqdm import tqdm

from src.utils import load_data, evaluate_predictions, save_results

# Load API key from env or .env file
try:
    from dotenv import load_dotenv
    load_dotenv('../.env')
except ImportError:
    pass

API_KEY = os.getenv('OPENROUTER_API_KEY', '')
if not API_KEY:
    raise EnvironmentError('Set OPENROUTER_API_KEY in your environment or .env file')

client = OpenAI(
    api_key=API_KEY,
    base_url='https://openrouter.ai/api/v1',
)
MODEL = 'deepseek/deepseek-chat'  # free tier on OpenRouter
print(f'Using model: {MODEL}')

## 1. Sample 200 stratified test reviews

In [ ]:
import random
random.seed(42)

test_texts, test_labels = load_data('test')
train_texts, train_labels = load_data('train')

# Stratified 200-sample subset
pos_idx = [i for i, l in enumerate(test_labels) if l == 'pos']
neg_idx = [i for i, l in enumerate(test_labels) if l == 'neg']

sample_idx = random.sample(pos_idx, 100) + random.sample(neg_idx, 100)
random.shuffle(sample_idx)

sample_texts  = [test_texts[i]  for i in sample_idx]
sample_labels = [test_labels[i] for i in sample_idx]

# 3 pos + 3 neg few-shot examples from train set (different from test)
few_shot_pos = [train_texts[i] for i in random.sample(
    [i for i, l in enumerate(train_labels) if l == 'pos'], 3)]
few_shot_neg = [train_texts[i] for i in random.sample(
    [i for i, l in enumerate(train_labels) if l == 'neg'], 3)]

print(f'Sample: {len(sample_texts)} reviews ({sum(l=="pos" for l in sample_labels)} pos, '
      f'{sum(l=="neg" for l in sample_labels)} neg)')

## 2. Prompt definitions

In [ ]:
def build_few_shot_examples():
    lines = []
    for p, n in zip(few_shot_pos, few_shot_neg):
        lines.append(f'[POS] {p[:300]}')
        lines.append(f'[NEG] {n[:300]}')
    return '\n'.join(lines)

FEW_SHOT_BLOCK = build_few_shot_examples()

PROMPTS = {
    'generic': lambda text: (
        'Classify the sentiment of the following text as "positive" or "negative".\n'
        'Reply with only one word.\n\n'
        f'Text: {text}'
    ),
    'domain': lambda text: (
        'You are analyzing movie reviews. Classify the sentiment of the following '
        'movie review as "positive" or "negative".\n'
        'Reply with only one word.\n\n'
        f'Review: {text}'
    ),
    'few_shot': lambda text: (
        'Classify movie review sentiment as "positive" or "negative".\n\n'
        f'Examples:\n{FEW_SHOT_BLOCK}\n\n'
        f'Now classify:\n{text}\n\n'
        'Reply with only one word: positive or negative.'
    ),
}

## 3. Inference helper

In [ ]:
def parse_label(response: str) -> str:
    """Extract pos/neg from model response."""
    r = response.strip().lower()
    if re.search(r'\bpositive\b', r):
        return 'pos'
    if re.search(r'\bnegative\b', r):
        return 'neg'
    # fallback: check first token
    first = r.split()[0] if r.split() else ''
    if first in ('pos', 'positive'):
        return 'pos'
    return 'neg'


def run_prompt_experiment(name: str, prompt_fn, texts, labels,
                          delay: float = 0.5, save_path: str = None):
    raw_responses = []
    preds = []
    errors = 0

    for text in tqdm(texts, desc=name):
        prompt = prompt_fn(text[:1500])  # truncate very long reviews
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=10,
                temperature=0,
            )
            answer = resp.choices[0].message.content or ''
        except Exception as e:
            answer = ''
            errors += 1
        raw_responses.append(answer)
        preds.append(parse_label(answer))
        time.sleep(delay)  # rate limiting

    print(f'{name}: {errors} API errors')

    # Save raw responses
    if save_path:
        df = pd.DataFrame({'text': texts, 'true': labels,
                           'response': raw_responses, 'pred': preds})
        df.to_csv(save_path, index=False)

    metrics = evaluate_predictions(labels, preds)
    return preds, metrics

## 4. Run all 3 prompts

In [ ]:
prompt_results = {}

for pname, pfn in PROMPTS.items():
    preds, metrics = run_prompt_experiment(
        name=f'Prompt:{pname}',
        prompt_fn=pfn,
        texts=sample_texts,
        labels=sample_labels,
        delay=0.3,
        save_path=f'../results/llm_{pname}_responses.csv',
    )
    prompt_results[pname] = metrics
    print(f'{pname}: {metrics}')
    save_results(
        task='2.4',
        approach=f'LLM ({MODEL}) — {pname} prompt',
        metrics=metrics,
        preprocessing='200-sample stratified subset, truncated to 1500 chars',
        notes=f'Model: {MODEL}, temperature=0'
    )

## 5. Comparison

In [ ]:
import matplotlib.pyplot as plt

comp = pd.DataFrame([
    {'Prompt': k, **v} for k, v in prompt_results.items()
])
display(comp.set_index('Prompt'))

ax = comp.set_index('Prompt')[['accuracy','precision','recall','f1']].plot(
    kind='bar', figsize=(8,4), rot=0, colormap='tab10')
ax.set_ylim(0, 1)
ax.set_title(f'Task 2.4 — LLM Prompting ({MODEL})')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/fig_llm_prompting.png', dpi=150, bbox_inches='tight')
plt.show()